In [2]:
import os
import uuid
import fitz                 
import pytesseract             
from PIL import Image         
from docx import Document         
import chromadb           
from chromadb.utils import embedding_functions
from openai import OpenAI

In [3]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = "/opt/homebrew/bin/tesseract"
print(pytesseract.get_tesseract_version())

5.5.2


In [4]:
print(os.getcwd())

/Users/sheetalsuwalka/Documents/Personal Projects/Rag System


In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

GEMINI_API_KEY=os.getenv("GEMINI_API_KEY")

EMBEDDING_MODEL     = "models/gemini-embedding-001"       # Gemini embedding model
LLM_MODEL           = "gemini-2.5-flash"  
CHROMA_COLLECTION   = "rag_documents"              # ChromaDB collection name
CHROMA_DB_PATH      = "./chroma_db"                # Local ChromaDB storage path
CHUNK_SIZE          = 600                          # Characters per chunk
CHUNK_OVERLAP       = 120                          # Overlap between chunks should be 20% of chunk size
TOP_K_RESULTS       = 8                           # Number of chunks to retrieve
DOCS_FOLDER = os.getcwd()


In [6]:
### check available emedding models
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

# List all available models that support embedContent
for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(model.name)

/Users/sheetalsuwalka/Documents/Personal Projects/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/y5/sqxvp96j7kq4n2yv8qbqmyc80000gn/T/ipykernel_65856/512305239.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-embedding-001


In [7]:
# Check embedding test
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

# Test embedding with a simple string
result = genai.embed_content(
    model=EMBEDDING_MODEL,
    content="Hello, this is a test",
    task_type="retrieval_document"
)

print(f"✅ Embedding works! Vector length: {len(result['embedding'])}")

✅ Embedding works! Vector length: 3072


In [60]:
result['embedding']

[-0.023655912,
 0.005955833,
 0.011001471,
 -0.09247737,
 -0.005719207,
 -0.003558516,
 -0.011125863,
 -0.004879417,
 0.0088616945,
 0.0028051194,
 -0.0067778723,
 -0.017225515,
 0.002823282,
 -0.003939283,
 0.16340183,
 0.0040177307,
 0.0002676556,
 0.0048556263,
 -0.0019964809,
 -0.009482187,
 -0.0021056936,
 -0.0083767045,
 0.009432797,
 0.0033966955,
 -0.0059444783,
 -0.0062793395,
 0.021757385,
 0.0048925355,
 0.015771681,
 -0.001678899,
 -0.0011669197,
 0.012024401,
 -0.0041618487,
 0.00797764,
 0.011536132,
 0.01408466,
 0.0059276796,
 0.005986926,
 -0.00810304,
 0.0064630597,
 -0.007003315,
 -0.009322294,
 -0.015341014,
 -0.008059883,
 0.020091185,
 0.012617928,
 -0.0003945933,
 -0.019743076,
 0.008738959,
 0.02109624,
 -0.015049705,
 0.005946386,
 -0.0151589615,
 -0.2938427,
 -0.012760457,
 0.005599606,
 -0.012413522,
 0.009864725,
 0.0055158916,
 -0.0043573477,
 -0.009885132,
 0.018379325,
 -0.015334697,
 -0.031859465,
 0.009885195,
 0.0071663125,
 0.003443843,
 0.0074281557,

In [43]:
import os

files = [
    f for f in os.listdir(DOCS_FOLDER)
    if f.lower().endswith((".pdf", ".png", ".jpg", ".jpeg", ".docx"))
]
print(f"Files found: {files}")

Files found: ['10. Driving License.pdf', 'Sheetal Data Intelligence Resume 2026.pdf', 'Aadhar Front.png', '8.  PAN Card.pdf', 'Self info.docx']


In [46]:
# ─────────────────────────────────────────────
# INITIALIZE CLIENTS
# ─────────────────────────────────────────────

import google.generativeai as genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# Custom Gemini Embedding Function for ChromaDB
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        embeddings = []
        for text in input:
            result = genai.embed_content(
                model=EMBEDDING_MODEL,
                content=text,
                task_type="retrieval_document"   # Use "retrieval_query" at query time
            )
            embeddings.append(result["embedding"])
        return embeddings

# ChromaDB with Gemini embedding function
gemini_ef = GeminiEmbeddingFunction()

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

chroma_client.delete_collection(CHROMA_COLLECTION)

collection = chroma_client.get_or_create_collection(
    name=CHROMA_COLLECTION,
    embedding_function=gemini_ef,
    metadata={"hnsw:space": "cosine"}
)

print(f"✅ Fresh collection created: {CHROMA_COLLECTION}\n")

✅ Fresh collection created: rag_documents



/var/folders/y5/sqxvp96j7kq4n2yv8qbqmyc80000gn/T/ipykernel_65856/1936318103.py:26: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  gemini_ef = GeminiEmbeddingFunction()


In [47]:
collections = chroma_client.list_collections()
print(f"Active collections: {[c.name for c in collections]}")

Active collections: ['rag_documents']


In [48]:

# ─────────────────────────────────────────────
# STEP 1: DOCUMENT PARSERS
# ─────────────────────────────────────────────

def parse_pdf(file_path: str) -> str:
    """Extract text from a PDF file using PyMuPDF."""
    text = ""
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text()
    print(f"  ✅ Parsed PDF: {file_path} ({len(text)} chars)")
    return text


def parse_png(file_path: str) -> str:
    """Extract text from an image using Tesseract OCR."""
    image = Image.open(file_path)
    text = pytesseract.image_to_string(image)
    print(f"  ✅ Parsed PNG (OCR): {file_path} ({len(text)} chars)")
    return text


def parse_docx(file_path: str) -> str:
    """Extract text from a DOCX file."""
    doc = Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
    print(f"  ✅ Parsed DOCX: {file_path} ({len(text)} chars)")
    return text


def parse_document(file_path: str) -> str:
    """Route to the correct parser based on file extension."""
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        return parse_pdf(file_path)
    elif ext in [".png", ".jpg", ".jpeg"]:
        return parse_png(file_path)
    elif ext == ".docx":
        return parse_docx(file_path)
    else:
        print(f"  ⚠️  Unsupported file type: {file_path}")
        return ""


In [49]:
parse_result = parse_document("Self info.docx")
print(parse_result[:100])

# import os

# files = [
#     f for f in os.listdir(DOCS_FOLDER)
#     if f.lower().endswith((".pdf", ".png", ".jpg", ".jpeg", ".docx"))
# ]

# for file_name in files:
#     file_path = os.path.join(DOCS_FOLDER, file_name)
#     text = parse_document(file_path)  # ✅ print inside will fire here
#     print(f"First 300 chars:\n{text[:300]}")
#     print("-" * 60)

  ✅ Parsed DOCX: Self info.docx (1393 chars)
Personal Detail:
Mr. Sheetal Suwalka, (Aadhar No. 443338290027), (PAN: IFOPS9307P) 
S/o Roshan Lal S


In [50]:
# ─────────────────────────────────────────────
# STEP 2: TEXT CHUNKING
# ─────────────────────────────────────────────

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Split text into overlapping chunks for better retrieval context.
    Overlap ensures that sentences split across chunk boundaries are still retrievable.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap  # Move forward with overlap
    return chunks

In [51]:
len(parse_result)

1393

In [52]:
chunks = chunk_text(parse_result)

print(f"Total chunks created: {len(chunks)}")
print(f"Chunk size config: {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars")
print("-" * 60)

Total chunks created: 3
Chunk size config: 600 chars | Overlap: 120 chars
------------------------------------------------------------


In [ ]:
# ─────────────────────────────────────────────
# STEP 3: LOAD DOCUMENTS & STORE IN CHROMADB
# ─────────────────────────────────────────────

def load_and_index_documents(docs_folder: str):
    """
    Parse all documents in the folder, chunk them,
    and store embeddings in ChromaDB.
    """
    if not os.path.exists(docs_folder):
        os.makedirs(docs_folder)
        print(f"📁 Created documents folder: {docs_folder}")
        print("   Please add your PDF, PNG, DOCX files there and re-run.")
        return

    files = [
        f for f in os.listdir(docs_folder)
        if f.lower().endswith((".pdf", ".png", ".jpg", ".jpeg", ".docx"))
        and not f.startswith("~$") 
    ]

    # When only changing specific doc
    # files = ["Self info.docx"]

    if not files:
        print("⚠️  No supported documents found in the folder.")
        return

    print(f"\n📂 Found {len(files)} document(s). Parsing and indexing...\n")

    for file_name in files:
        file_path = os.path.join(docs_folder, file_name)

        # Parse the document
        raw_text = parse_document(file_path)
        if not raw_text.strip():
            print(f"  ⚠️  No text extracted from {file_name}, skipping.")
            continue

        # Chunk the text
        chunks = chunk_text(raw_text)
        print(f"  📄 {file_name} → {len(chunks)} chunks")

        # Prepare data for ChromaDB
        ids        = [str(uuid.uuid4()) for _ in chunks]
        metadatas  = [{"source": file_name, "chunk_index": i} for i, _ in enumerate(chunks)]

        # Store in ChromaDB (embeddings are auto-generated by openai_ef)
        collection.add(documents=chunks, metadatas=metadatas, ids=ids)

    print(f"\n✅ Indexing complete! Total chunks in DB: {collection.count()}\n")
    

In [54]:
load_and_index_documents(DOCS_FOLDER)


📂 Found 5 document(s). Parsing and indexing...

  ✅ Parsed PDF: /Users/sheetalsuwalka/Documents/Personal Projects/Rag System/10. Driving License.pdf (532 chars)
  📄 10. Driving License.pdf → 2 chunks
  ✅ Parsed PDF: /Users/sheetalsuwalka/Documents/Personal Projects/Rag System/Sheetal Data Intelligence Resume 2026.pdf (4249 chars)
  📄 Sheetal Data Intelligence Resume 2026.pdf → 9 chunks
  ✅ Parsed PNG (OCR): /Users/sheetalsuwalka/Documents/Personal Projects/Rag System/Aadhar Front.png (393 chars)
  📄 Aadhar Front.png → 1 chunks
  ✅ Parsed PDF: /Users/sheetalsuwalka/Documents/Personal Projects/Rag System/8.  PAN Card.pdf (205 chars)
  📄 8.  PAN Card.pdf → 1 chunks
  ✅ Parsed DOCX: /Users/sheetalsuwalka/Documents/Personal Projects/Rag System/Self info.docx (1393 chars)
  📄 Self info.docx → 3 chunks

✅ Indexing complete! Total chunks in DB: 16



In [ ]:
results = collection.get(limit=5,
    include=["embeddings", "documents", "metadatas"])

print(f"Document : {results['documents'][4][:200]}")
print(f"Source   : {results['metadatas'][4]['source']}")
print(f"Embedding vector length : {len(results['embeddings'][4])}")
print(f"First 5 values of vector: {results['embeddings'][4][:15]}")

Document : s issues like Shipments ageing, Trips delay and warehouse performance.
Network & Operations Analytics
Improved logistics network speed by 5% by identifying bottleneck cohorts, segmenting transit phase
Source   : Sheetal Data Intelligence Resume 2026.pdf
Embedding vector length : 3072
First 5 values of vector: [ 0.00359659  0.00471924  0.01083077 -0.07148866 -0.01282006 -0.00049458
 -0.00433335  0.01394504 -0.01657411 -0.02459111  0.0105757   0.01226887
 -0.00173395  0.04526362  0.10185529]


In [ ]:
# ─────────────────────────────────────────────
# STEP 4: RETRIEVAL
# ─────────────────────────────────────────────

def retrieve_relevant_chunks(query: str, top_k: int = TOP_K_RESULTS) -> list[dict]:

    # Generate query embedding with correct task type
    query_embedding = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=query,
        task_type="retrieval_query"   
    )["embedding"]

    # Internally ChromaDB does this:
    # for each stored chunk:
    #     similarity = cosine_similarity(query_vector, chunk_vector)
    # return top 5 highest similarity scores

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k    )

    chunks = []
    for i, doc in enumerate(results["documents"][0]):
        chunks.append({
            "text": doc,
            "source": results["metadatas"][0][i]["source"],
            "chunk_index": results["metadatas"][0][i]["chunk_index"]
        })

    return chunks

In [ ]:
# ─────────────────────────────────────────────
# STEP 5: GENERATION (RAG)
# ─────────────────────────────────────────────

def ask_question(question: str) -> str:
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks from ChromaDB
    2. Build a prompt with the context
    3. Send to Gemini and return the answer
    """
    print(f"\n🔍 Question: {question}")

    # Retrieve relevant context
    relevant_chunks = retrieve_relevant_chunks(question)

    if not relevant_chunks:
        return "❌ No relevant information found in the documents."

    # Build context string
    context = ""
    for i, chunk in enumerate(relevant_chunks):
        context += f"\n--- Chunk {i+1} (Source: {chunk['source']}) ---\n{chunk['text']}\n"

    # Print sources used
    sources = list(set([c["source"] for c in relevant_chunks]))
    print(f"📎 Sources used: {', '.join(sources)}")

    # Build the prompt
    prompt = f"""You are a helpful assistant that answers questions based on the provided context.
### Rules:
- Answer ONLY from the context provided
- If answer is not in context, say "I don't have enough information"
- Be specific and detailed in your answers
- Always mention which document the answer came from
- If the question is about a person, extract all relevant details

Context from documents:
{context}

Question: {question}

Instructions:
- Give a detailed answer
- Cite which document each piece of information came from
- If multiple documents are relevant, combine the information """
    
    # return prompt

    # Call Gemini LLM
    model = genai.GenerativeModel(LLM_MODEL)
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0.2,
            max_output_tokens=700
        )
    )

    return response.text

In [110]:
# print(ask_question("For work experience of sheetal in GenAI & Automation Projects, give any project highlight "))

In [58]:
# question = "How Address of sheetal changed with time"
# question = "What is aadhar number of Sheetal"
# question = "For work experience of sheetal in GenAI & Automation Projects, give any project highlight "
question = "Give the vehicle information like which bike sheetal have"


In [59]:
print(ask_question(question))


🔍 Question: Give the vehicle information like which bike sheetal have
📎 Sources used: Aadhar Front.png, Self info.docx, 8.  PAN Card.pdf, Sheetal Data Intelligence Resume 2026.pdf, 10. Driving License.pdf
Sheetal Suwalka owns a Royal Enfield Meteor 350 bike. It was purchased in December 2023, and its bike number is HR 98 N 5218.

(Source: Self info.docx)


In [41]:
retrieved_info = retrieve_relevant_chunks(question)
print(f"📎 Retrieved {len(retrieved_info)} chunks:\n")
for i, chunk in enumerate(retrieved_info):
    print(f"--- Chunk {i+1} | Source: {chunk['source']} ---")
    print(chunk['text'])
    print("-" * 60)

📎 Retrieved 8 chunks:

--- Chunk 1 | Source: Self info.docx ---
Personal Detail:
Mr. Sheetal Suwalka, (Aadhar No. 443338290027), (PAN: IFOPS9307P) 
S/o Roshan Lal Suwalka, Behind
Hospital, Rashmi, District- Chittaurgarh, Rajasthan 312203, 
HDFC Account No.: 50100362503248
Email Address: SHEETALSUWALKA324@GMAIL.COM
Phone no: 8209852472
Residential Address: 
From March 2020- April 2022
Sheetal Villa, Behind Hospital, Rashmi, Chittaurgarh, Rajasthan - 312203
From May 2022 – June 2023:
698, Sector 3, Near JMB, Sector 3, Udaipur, Rajasthan - 313002
From July 2023 – April 2024:
C1831, 3rd Floor Sushant Lok Sector -43 Phase -1 Gurugram, Haryana- 122009,
From May
------------------------------------------------------------
--- Chunk 2 | Source: Self info.docx ---
Personal Detail:
Mr. Sheetal Suwalka, (Aadhar No. 443338290027), (PAN: IFOPS9307P) S/o Roshan Lal Suwalka, Behind
Hospital, Rashmi, District- Chittaurgarh, Rajasthan 312203, 
HDFC Account No.: 50100362503248
Email Address: : SHEETALSU